In [14]:
ord("h")

104

In [15]:
ord("ក") # take only one input 



6016

In [16]:
[ord(x) for x in "សួស្តីបងប្រុស"]

[6047, 6077, 6047, 6098, 6031, 6072, 6036, 6020, 6036, 6098, 6042, 6075, 6047]

In [17]:
[ord(x) for x in "Hello brother"]

[72, 101, 108, 108, 111, 32, 98, 114, 111, 116, 104, 101, 114]

In [ ]:
from pathlib import Path

CORPUS_NAME = "khmer_train_small.txt"
MAX_CHARS = None

candidates = [Path.cwd() / CORPUS_NAME, Path.cwd() / "token2" / CORPUS_NAME]
CORPUS = next((c for c in candidates if c.exists()), None)
if CORPUS is None:
    raise FileNotFoundError(
        f"{CORPUS_NAME} not found. Looked in:\n  "
        + "\n  ".join(str(c) for c in candidates)
    )

text = CORPUS.read_text(encoding="utf-8")
if MAX_CHARS:
    text = text[:MAX_CHARS]

print(f"{CORPUS.name}: {len(text):,} chars -> {len(text.encode('utf-8')):,} utf-8 bytes")

In [19]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

# a map is 

In [20]:
len(tokens)

5794024

In [21]:
def get_stats(ids):
    counts = {} 
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair,0) + 1
    return counts 


In [22]:
stats = get_stats(tokens)

top_pair = max(stats, key = stats.get)

print(top_pair)

count = stats[top_pair]

(225, 158)


In [23]:
raw_bytes = bytes([top_pair[0], top_pair[1]])
print(f"Top pair: {top_pair} -> {raw_bytes} (count: {count})")

Top pair: (225, 158) -> b'\xe1\x9e' (count: 1431687)


In [24]:
print(f"Decoded: {raw_bytes.decode('utf-8', errors='replace')}")

Decoded: �


In [25]:
def merge(ids, pair, idx):
    # ids:  the current list of token integers (e.g. [1, 2, 3, 1, 2])
    # pair: the tuple pair we want to replace (e.g. (1, 2))
    # idx:  the new token ID to replace the pair with (e.g. 256)
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2 
        else:
            new_ids.append(ids[i])
            i +=1 
    
    return new_ids

In [26]:
vocab_size = 3000
num_merges = vocab_size - 256
ids = list(tokens)

merges = {} # (int,int) 205,206 -> int 257

for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key= stats.get) # top pair of ids 
    idx = 256 + i
    ids = merge(ids, pair, idx) # one merge from the merge function 
    merges[pair] = idx

    

In [27]:
print(f"characters:        {len(text):,}")
print(f"bytes (tokens):    {len(tokens):,}")
print(f"after merges (ids):{len(ids):,}")
print(f"byte compression:  {len(tokens) / len(ids):.2f}X")
print(f"chars/token:       {len(text) / len(ids):.3f}")


characters:        1,999,635
bytes (tokens):    5,794,024
after merges (ids):668,979
byte compression:  8.66X
chars/token:       2.989


In [28]:
vocab = {idx: bytes([idx]) for idx in range(256)} # ths is just looping all of the eng stuff

for (p0,p1),idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text 


In [38]:
def encode(text):
    tokens = list(text.encode("utf-8")) # convert into raw bytes  
    while len(tokens) >= 2 :
        stats = get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf"))) # take the lowest index inside the merge
        if pair not in merges:
            break # if no pair prevent from the value with inf getting return
        idx = merges[pair]
        tokens = merge(tokens,pair,idx)
    return tokens
        

In [40]:
decode(encode("លោក"))

'លោក'

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

In [41]:
len(vocab)

3000